## P14_P22_book

In [ ]:
import json, math, re
from pathlib import Path

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.models.resnet import ResNet, BasicBlock

RUNS = Path("data/resnet/hackin/aug_17_run")                       # <-- output dir of train_depth_sweep.py
DATA = Path("data/imagenet")     # <-- edit
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
MODELS = [m for m in MODEL_DEPTHS if (RUNS / m).exists()]

def color_of(name):
    """Plain nets: dark->light with depth. ResNet: red."""
    if name == "resnet74":
        return "crimson"
    depths = sorted(d for m, d in MODEL_DEPTHS.items() if m != "resnet74")
    return plt.cm.viridis(depths.index(MODEL_DEPTHS[name]) / (len(depths) - 1))

print(torch.__version__, device, "| runs found:", MODELS)

In [ ]:
# --- model definitions (mirrors train_depth_sweep.py) ---
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def blocks_of(model):
    return [b for s in (model.layer1, model.layer2, model.layer3, model.layer4)
            if isinstance(s, nn.Sequential) for b in s]

In [ ]:
# --- metrics loading: prefer metrics.npz, fall back to metrics.jsonl ---
def load_metrics(name):
    d = RUNS / name
    if (d / "metrics.npz").exists():
        return dict(np.load(d / "metrics.npz"))
    recs = [json.loads(l) for l in open(d / "metrics.jsonl") if l.strip()]
    out = {}
    for k in recs[0]:
        if k in ("block_grad_norms", "block_acts"):
            continue
        out[k] = np.array([r.get(k, np.nan) for r in recs], dtype=np.float64)
    out["block_grad_norms"] = np.array([r["block_grad_norms"] for r in recs])
    return out

M = {name: load_metrics(name) for name in MODELS}
for name in MODELS:
    print(f"{name:9s} records={len(M[name]['step']):5d} "
          f"last step={int(M[name]['step'][-1])}")

In [ ]:
def list_ckpts(name):
    return sorted(int(re.search(r"(\d+)", p.stem).group(1))
                  for p in (RUNS / name).glob("ckpt_step*.pt"))

def load_model(name, step=None):
    d = RUNS / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

print("plain74 checkpoints:", list_ckpts("plain74")[:5], "...",
      list_ckpts("plain74")[-2:])

In [ ]:
# --- validation data (same transforms as training script) ---
import torchvision.datasets as dsets, torchvision.transforms as T

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_ds = dsets.ImageFolder(DATA / "ILSVRC/Data/CLS-LOC/val",
    T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
               T.Normalize(MEAN, STD)]))
val_loader = torch.utils.data.DataLoader(val_ds, 64, shuffle=True,
    num_workers=4, pin_memory=True)

# wnid -> human-readable name (Kaggle CLS-LOC ships this mapping)
wnid_words = {}
mapping = DATA / "LOC_synset_mapping.txt"
if mapping.exists():
    for line in open(mapping):
        wnid, words = line.strip().split(" ", 1)
        wnid_words[wnid] = words.split(",")[0]

def class_name(idx):
    if 0 <= idx < len(val_ds.classes):
        wnid = val_ds.classes[idx]
        return wnid_words.get(wnid, wnid)
    return f"cls{idx}"

def denorm(x):  # CHW tensor -> HWC numpy in [0,1] for imshow
    img = x.cpu() * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

print(len(val_ds), "val images,", len(val_ds.classes), "classes")

In [ ]:
# --- forward + backward on a batch of val images ---
name = "plain74" if "plain74" in MODELS else MODELS[-1]
model = load_model(name)

x, y = next(iter(val_loader))
x, y = x[:16].to(device), y[:16].to(device)

logits = model(x)
loss = F.cross_entropy(logits, y)
model.zero_grad(set_to_none=True)
loss.backward()

probs = logits.softmax(1)
top5p, top5i = probs.topk(5, dim=1)
print(f"{name}: batch loss={loss.item():.3f}, "
      f"batch top-1={(logits.argmax(1) == y).float().mean().item():.3f}\n")
for i in range(3):
    guesses = ", ".join(f"{class_name(top5i[i,j].item())} {top5p[i,j]:.2f}"
                        for j in range(5))
    print(f"true: {class_name(y[i].item()):22s} -> {guesses}")

## Intermediate Activations

In [ ]:
# --- cache all intermediate activations through a model ---
class ActivationCache:
    """with ActivationCache(model) as cache: model(x)  ->  cache["layer3.5.bn2"]
    Hooks every leaf module (+ the residual blocks and stages themselves),
    stores detached outputs keyed by qualified module name, in forward order."""
    def __init__(self, model, to_cpu=True, leaf_only=False):
        self.model, self.to_cpu, self.leaf_only = model, to_cpu, leaf_only
        self.cache, self.handles = {}, []

    def _keep(self, name, mod):
        if name == "":                      # the model itself
            return False
        if self.leaf_only:
            return len(list(mod.children())) == 0
        return not isinstance(mod, (nn.Sequential, nn.Identity))

    def __enter__(self):
        def hook(name):
            def fn(mod, inp, out):
                if torch.is_tensor(out):
                    self.cache[name] = out.detach().cpu() if self.to_cpu else out.detach()
            return fn
        for name, mod in self.model.named_modules():
            if self._keep(name, mod):
                self.handles.append(mod.register_forward_hook(hook(name)))
        return self.cache

    def __exit__(self, *a):
        for h in self.handles:
            h.remove()
        self.handles.clear()

# name = "plain74" if "plain74" in MODELS else MODELS[-1]
name = "plain14"
model = load_model(name)
x, y = next(iter(val_loader))
x = x[:8].to(device)

with ActivationCache(model) as cache:
    with torch.no_grad():
        model(x)

total = sum(v.numel() * v.element_size() for v in cache.values()) / 1e9
print(f"{name}: cached {len(cache)} activations, {total:.2f} GB "
      f"for batch of {x.shape[0]}\n")
for k in list(cache)[:6]:
    print(f"{k:20s} {tuple(cache[k].shape)}")
print("...")
# e.g. mean |activation| out of each residual block, through the net:
block_keys = [k for k in cache if k.count(".") == 1 and k.startswith("layer")]
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot([cache[k].abs().mean().item() for k in block_keys], "o-",
        color=color_of(name))
ax.set(xlabel="block index", ylabel="mean |activation|",
       title=f"{name}: activation magnitude through the network")
ax.grid(alpha=0.3); plt.tight_layout()

In [ ]:
block_keys

In [ ]:
cache.keys()

In [ ]:
# for k, v in cache.items():
#     print(k, v.shape)

## Activations
- Examples we're getting right/wrong
- One will be the Drum
- This is for ResNet-8

In [ ]:
def load_image(idx):
    x, y = val_ds[idx]
    return x.unsqueeze(0).to(device), torch.tensor([y], device=device)

In [ ]:
@torch.no_grad()
def correct_ans_conf(model, x, y):
    return torch.nn.Softmax()(model(x))[0, y.item()].item()

In [ ]:
model=load_model('plain8')

In [ ]:
save_dir = Path('data/resnet/hackin/p19_samples')
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
rI = np.arange(50000)
np.random.shuffle(rI)

In [ ]:
# for img_idx in rI[:1000]:

#     x, y = load_image(img_idx)
#     cls = val_ds.classes[y.item()]
#     # print(wnid_words[cls], correct_ans_conf(model, x, y))
#     yhat_index=model(x).argmax().item()
    
#     plt.imshow(denorm(x[0]))
#     plt.title(wnid_words[cls] + ', ' + wnid_words[val_ds.classes[yhat_index]])
#     plt.savefig(save_dir/(str(img_idx)+'.png'))

Nice correct examples
 - `47587` Lemon
 - `6538` Flamingo
 - `21299` Barn

Nice incorrect examples
- `38250` Rocking chair -> turnstyle
- `38552` safe -> rotisserie

In [ ]:
img_idx=38250

x, y = load_image(img_idx)
cls = val_ds.classes[y.item()]
print(wnid_words[cls], correct_ans_conf(model, x, y))
yhat_index=model(x).argmax().item()

from PIL import Image
img = Image.fromarray((denorm(x[0])*255).astype('uint8'))
# img.save("data/resnet/hackin/p20/rocking_chair.jpg")

In [ ]:
val_ds.samples[20907][0]

In [ ]:
img_idx=47587
# img_idx=38552

x, y = load_image(img_idx)
cls = val_ds.classes[y.item()]
print(wnid_words[cls], correct_ans_conf(model, x, y))
yhat_index=model(x).argmax().item()

from PIL import Image
img = Image.fromarray((denorm(x[0])*255).astype('uint8'))
# img.save("data/resnet/hackin/p20/safe.jpg")

In [ ]:
with ActivationCache(model) as cache:
    with torch.no_grad():
        model(x)

In [ ]:
for k, v in cache.items():
    print(k, v.shape)

In [ ]:
cache_np={k:v.cpu().numpy() for k,v in cache.items()}

In [ ]:
for k, v in cache_np.items():
    print(k, v.shape)

In [ ]:
cache_np['image']=denorm(x[0])

In [ ]:
plt.imshow(denorm(x[0]))

In [ ]:
# np.save('data/resnet/hackin/p13/lemon_activations_47587', cache_np)

In [ ]:
all_params={k:v for k, v in model.named_parameters()}

In [ ]:
all_params.keys()

In [ ]:
all_params['conv1.weight'].shape

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Define white/black to pure magenta ('#ff00ff')
magenta_cmap = LinearSegmentedColormap.from_list(
    'magenta_intensity', ['#000000', '#ff00ff']
)

In [ ]:
# plt.imshow(all_params['conv1.weight'][0].detach().cpu().permute(1, 2, 0))
# plt.imshow(denorm(all_params['conv1.weight'][22].detach()))
fig=plt.figure(0,(5,5), facecolor='k')
plt.imshow(all_params['conv1.weight'][22, 0].detach().cpu(), cmap=magenta_cmap)
plt.axis('off')
# plt.savefig('data/resnet/hackin/p13/k20.png')

## Book Renders

In [ ]:
fig=plt.figure(0, (6,6))
count=0
for i in range(8):
    for j in range(8):
        ax=fig.add_subplot(8,8,count+1)
        ax.imshow(denorm(all_params['conv1.weight'][count].detach().cpu()))
        count+=1

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Define white/black to pure magenta ('#ff00ff')
ocean_cmap = LinearSegmentedColormap.from_list(
    'orange_intensity', ['#000000', '#EB8423'] ##EB8423 F7931E
)

fig=plt.figure(0,(10,5)) #, facecolor='k')
for i in range(3):
    ax=fig.add_subplot(1,3,i+1)
    x=all_params['conv1.weight'][22, :].detach().cpu()
    plt.imshow(x[i], vmin=x.min(), vmax=x.max(), cmap=ocean_cmap)
    plt.axis('off')

plt.colorbar()

# plt.savefig('data/resnet/hackin/k20_orange_2.png')

In [ ]:
all_params['conv1.weight'][22, :].shape

In [ ]:
# img_idx=20907

# x, y = load_image(img_idx)
# cls = val_ds.classes[y.item()]
# print(wnid_words[cls], correct_ans_conf(model, x, y))
# yhat_index=model(x).argmax().item()

# plt.imshow(denorm(x[0]))

In [ ]:
# np.save('data/resnet/hackin/p13/plain_8_conv_1', 
#         all_params['conv1.weight'].detach().cpu().numpy())

In [ ]:
plt.imshow(denorm(x[0]))

In [ ]:
denorm(x[0]).max()

In [ ]:
from PIL import Image
img = Image.fromarray((denorm(x[0])*255).astype('uint8'))
# img.save("data/resnet/hackin/p13/lemon.jpg")

In [ ]:
cache_np['conv1'].shape

In [ ]:
fig=plt.figure(0, (14,14))
count=0
for i in range(8):
    for j in range(8):
        ax=fig.add_subplot(8,8,count+1)
        ax.imshow(cache_np['conv1'][0, count])
        count+=1

In [ ]:
# out_dir='data/resnet/hackin/p13/conv_1_activations'
# Path(out_dir).mkdir(parents=True, exist_ok=True)
# cmap=plt.get_cmap('viridis')

# for c in range(cache_np['conv1'][0].shape[0]):
#     v=cache_np['conv1'][0][c]
#     v=(v-v.min())/(v.max()-v.min())   #imshow's default per-image min-max stretch
#     rgb=(cmap(v)[..., :3]*255).astype(np.uint8)
#     Image.fromarray(rgb).resize((10*v.shape[1], 10*v.shape[0]),
#                                 Image.NEAREST).save(f'{out_dir}/act_{c:02d}.png')#

## Colorbar for book

In [ ]:
plt.imshow(cache_np['conv1'][0, 0])
plt.colorbar()
# plt.savefig('data/resnet/hackin/colorbar_for_book_1.png', dpi=400)

In [ ]:
cache_np['conv1'][0, :].max()

In [ ]:
cache_np['conv1'][0, :].min()

In [ ]:
fig=plt.figure(0, (14,14))
count=0
for i in range(8):
    for j in range(8):
        ax=fig.add_subplot(8,8,count+1)
        ax.imshow(cache_np['layer1.0.conv1'][0, count])
        count+=1

In [ ]:
out_dir='data/resnet/hackin/p13/conv_2_activations'
Path(out_dir).mkdir(parents=True, exist_ok=True)
cmap=plt.get_cmap('viridis')

for c in range(cache_np['layer1.0.conv1'][0].shape[0]):
    v=cache_np['layer1.0.conv1'][0][c]
    v=(v-v.min())/(v.max()-v.min())   #imshow's default per-image min-max stretch
    rgb=(cmap(v)[..., :3]*255).astype(np.uint8)
    Image.fromarray(rgb).resize((10*v.shape[1], 10*v.shape[0]),
                                Image.NEAREST).save(f'{out_dir}/act_{c:02d}.png')

In [ ]:
np.save('data/resnet/hackin/p13/plain_8_layer1_0_conv1.npy', 
        all_params['layer1.0.conv1.weight'].detach().cpu().numpy())

In [ ]:
all_params.keys()

In [ ]:
cache_np.keys()

In [ ]:
plt.plot(cache_np['fc'].ravel())

In [ ]:
save_dir

In [ ]:
fig=plt.figure(0, (2,10))
plt.plot(cache_np['fc'].ravel(), np.arange(1000))
plt.ylim([1010, -10])
plt.savefig('data/ai_book_vol_2/3_resnets/graphics/lemon_probs.svg')

In [ ]:
plt.plot(torch.nn.Softmax(1)(cache['fc']).ravel())

In [ ]:
# cache['fc']

In [ ]:
cache['fc'].shape

In [ ]:
vals, ind = torch.topk(cache['fc'].ravel(), 50)

In [ ]:
vals, ind

In [ ]:
for v, i in zip(vals, ind):
    print(v,i, class_name(i))

# P22

In [ ]:
model=load_model('plain14')

with ActivationCache(model) as cache:
    with torch.no_grad():
        model(x)

cache_np={k:v.cpu().numpy() for k,v in cache.items()}
cache_np['image']=denorm(x[0])

In [ ]:
save_dir=Path('data/resnet/hackin/p22/')
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# np.save(save_dir/'lemon_activations_47587_plain14', cache_np)

In [ ]:
for k, v in cache_np.items():
    print(k, v.shape)